# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gullahmadbhatti0155/MLtask1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [11]:
import getpass
import os

if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass.getpass("Paste HF Token: ")

In [17]:
import urllib.request

try:
    urllib.request.urlopen("https://huggingface.co", timeout=5)
    print("Internet connection to Hugging Face is working!")
except Exception as e:
    print("Network Connection Failed:", e)

Internet connection to Hugging Face is working!


In [20]:
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files(repo_id="FlyRank/internship-warehouse", repo_type="dataset")

print("Files in repository:")
for f in files:
    print(" -", f)

Files in repository:
 - .gitattributes
 - README.md
 - dim_clients.parquet
 - dim_content.parquet
 - fact_content_daily_performance/month=2025-01/data_0.parquet
 - fact_content_daily_performance/month=2025-02/data_0.parquet
 - fact_content_daily_performance/month=2025-03/data_0.parquet
 - fact_content_daily_performance/month=2025-04/data_0.parquet
 - fact_content_daily_performance/month=2025-05/data_0.parquet
 - fact_content_daily_performance/month=2025-06/data_0.parquet
 - fact_content_daily_performance/month=2025-07/data_0.parquet
 - fact_content_daily_performance/month=2025-08/data_0.parquet
 - fact_content_daily_performance/month=2025-09/data_0.parquet
 - fact_content_daily_performance/month=2025-10/data_0.parquet
 - fact_content_daily_performance/month=2025-11/data_0.parquet
 - fact_content_daily_performance/month=2025-12/data_0.parquet
 - fact_content_daily_performance/month=2026-01/data_0.parquet
 - fact_content_daily_performance/month=2026-02/data_0.parquet
 - fact_content_dail

In [25]:
import os
import duckdb
from huggingface_hub import hf_hub_download, HfApi

# 1. Fetch file list and filter SPECIFICALLY for March 2026 partition
api = HfApi()
repo_files = api.list_repo_files(repo_id="FlyRank/internship-warehouse", repo_type="dataset")

march_files = [
    f for f in repo_files 
    if "fact_content_daily_performance" in f 
    and "month=2026-03" in f 
    and f.endswith(".parquet")
]

# Fallback: If Hive partition naming format differs, grab the latest parquet file
if not march_files:
    march_files = [f for f in repo_files if "fact_content_daily_performance" in f and f.endswith(".parquet")][-1:]

print(f"Targeting partition file: {march_files[0]}")

# 2. Download ONLY March 2026 file (~15-20MB instead of hundreds of MBs)
local_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename=march_files[0],
    repo_type="dataset"
)

# 3. Execute Verification Queries in DuckDB
con = duckdb.connect()

print("\n--- Query 1: Uniqueness Check ---")
q1 = f"""
SELECT 
    client_hash_id, 
    content_hash_id, 
    report_date,
    COUNT(*) AS row_count
FROM '{local_file}'
GROUP BY client_hash_id, content_hash_id, report_date
HAVING COUNT(*) > 1
LIMIT 5;
"""
print(con.execute(q1).df())

print("\n--- Query 2: Row Count & Date Span ---")
q2 = f"""
SELECT 
    COUNT(*) AS total_rows,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM '{local_file}';
"""
print(con.execute(q2).df())

print("\n--- Query 3: Availability Filtering ---")
q3 = f"""
SELECT 
    COUNT(*) AS total_rows,
    COUNTIF(client_has_ga4 IS TRUE) AS ga4_survived_rows,
    COUNTIF(client_has_gsc IS TRUE) AS gsc_survived_rows
FROM '{local_file}';
"""
print(con.execute(q3).df())

Targeting partition file: fact_content_daily_performance/month=2026-03/data_0.parquet

--- Query 1: Uniqueness Check ---


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Empty DataFrame
Columns: [client_hash_id, content_hash_id, report_date, row_count]
Index: []

--- Query 2: Row Count & Date Span ---
   total_rows   min_date   max_date
0     9841378 2026-03-01 2026-03-31

--- Query 3: Availability Filtering ---
   total_rows  ga4_survived_rows  gsc_survived_rows
0     9841378          6822637.0          9841378.0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [26]:
import duckdb

con = duckdb.connect()

# Inspect column names and data types from the local March 2026 file
schema_df = con.execute(f"DESCRIBE SELECT * FROM '{local_file}';").df()
print(schema_df[['column_name', 'column_type']])

                 column_name column_type
0                report_date        DATE
1             client_hash_id     VARCHAR
2            content_hash_id     VARCHAR
3             client_has_gsc     BOOLEAN
4             client_has_ga4     BOOLEAN
5         gsc_data_available     BOOLEAN
6         ga4_data_available     BOOLEAN
7            gsc_impressions      BIGINT
8                 gsc_clicks      BIGINT
9           gsc_sum_position      BIGINT
10          gsc_avg_position      DOUBLE
11             ga4_pageviews      BIGINT
12              ga4_sessions      BIGINT
13                 ga4_users      BIGINT
14      ga4_engaged_sessions      BIGINT
15  ga4_total_engagement_sec      BIGINT
16          sessions_organic      BIGINT
17           sessions_direct      BIGINT
18         sessions_referral      BIGINT
19           sessions_social      BIGINT
20             sessions_paid      BIGINT
21               sessions_ai      BIGINT
22                ai_chatgpt      BIGINT
23             a

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [27]:
import duckdb

con = duckdb.connect()

print("==================================================")
print(" 1. GRAIN CHECK (client_hash_id + content_hash_id + report_date)")
print("==================================================")
q_grain = f"""
SELECT 
    client_hash_id, 
    content_hash_id, 
    report_date, 
    COUNT(*) as dup_count
FROM '{local_file}'
GROUP BY client_hash_id, content_hash_id, report_date
HAVING COUNT(*) > 1;
"""
df_grain = con.execute(q_grain).df()
print(f"Duplicate Grain Records Found: {len(df_grain)}")
print(df_grain)

print("\n==================================================")
print(" 2. ROW COUNTS & DATE WINDOW CHECK")
print("==================================================")
q_window = f"""
SELECT 
    COUNT(*) AS total_rows,
    MIN(report_date) AS window_start,
    MAX(report_date) AS window_end,
    COUNT(DISTINCT report_date) AS total_active_days
FROM '{local_file}';
"""
print(con.execute(q_window).df())

print("\n==================================================")
print(" 3. MISSING VALUES / NULL CHECKS FOR KEY FIELDS")
print("==================================================")
q_nulls = f"""
SELECT 
    COUNT(*) - COUNT(client_hash_id) AS null_clients,
    COUNT(*) - COUNT(content_hash_id) AS null_contents,
    COUNT(*) - COUNT(report_date) AS null_dates,
    COUNT(*) - COUNT(gsc_clicks) AS null_gsc_clicks,
    COUNT(*) - COUNT(ga4_sessions) AS null_ga4_sessions
FROM '{local_file}';
"""
print(con.execute(q_nulls).df())

print("\n==================================================")
print(" 4. INTEGRATION AVAILABILITY BREAKDOWN")
print("==================================================")
q_avail = f"""
SELECT 
    COUNTIF(gsc_data_available IS TRUE) AS gsc_available_count,
    COUNTIF(ga4_data_available IS TRUE) AS ga4_available_count,
    ROUND(COUNTIF(gsc_data_available IS TRUE) * 100.0 / COUNT(*), 2) AS gsc_available_pct,
    ROUND(COUNTIF(ga4_data_available IS TRUE) * 100.0 / COUNT(*), 2) AS ga4_available_pct
FROM '{local_file}';
"""
print(con.execute(q_avail).df())

 1. GRAIN CHECK (client_hash_id + content_hash_id + report_date)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate Grain Records Found: 0
Empty DataFrame
Columns: [client_hash_id, content_hash_id, report_date, dup_count]
Index: []

 2. ROW COUNTS & DATE WINDOW CHECK
   total_rows window_start window_end  total_active_days
0     9841378   2026-03-01 2026-03-31                 31

 3. MISSING VALUES / NULL CHECKS FOR KEY FIELDS
   null_clients  null_contents  null_dates  null_gsc_clicks  null_ga4_sessions
0             0              0           0                0            3018741

 4. INTEGRATION AVAILABILITY BREAKDOWN
   gsc_available_count  ga4_available_count  gsc_available_pct  \
0            3611061.0             413966.0              36.69   

   ga4_available_pct  
0               4.21  


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

GSC-Only Early Rows & Missing GA4: GSC coverage is 100%, but GA4 is only ~69.3% available. The dataset cannot provide user engagement (ga4_sessions, scroll_events) for the ~30.7% of non-GA4 rows.

Unbalanced History & Onboarding Bias: Single-month scope cannot reveal historical trends or determine whether missing GA4 rows stem from recent client onboarding, tracking dropouts, or API delays.

Window Overlaps & Attribution Discrepancy: GSC records search-engine event timestamps, while GA4 records client-side session starts. Timezone and tracking window differences mean GSC clicks and GA4 organic sessions won't align 1:1 daily.

In [28]:
import duckdb

con = duckdb.connect()

print("==================================================")
print(" 1. GSC-ONLY vs. DUAL-INTEGRATION GAP")
print("==================================================")
q_limit_integration = f"""
SELECT 
    COUNTIF(gsc_data_available IS TRUE AND ga4_data_available IS FALSE) AS gsc_only_rows,
    COUNTIF(gsc_data_available IS TRUE AND ga4_data_available IS TRUE) AS dual_integration_rows,
    ROUND(COUNTIF(gsc_data_available IS TRUE AND ga4_data_available IS FALSE) * 100.0 / COUNT(*), 2) AS gsc_only_pct
FROM '{local_file}';
"""
print(con.execute(q_limit_integration).df())

print("\n==================================================")
print(" 2. CLIENT-LEVEL UNBALANCED COVERAGE")
print("==================================================")
q_limit_clients = f"""
SELECT 
    client_hash_id,
    COUNT(*) AS total_records,
    COUNTIF(ga4_data_available IS TRUE) AS ga4_records,
    ROUND(COUNTIF(ga4_data_available IS TRUE) * 100.0 / COUNT(*), 2) AS ga4_coverage_pct
FROM '{local_file}'
GROUP BY client_hash_id
ORDER BY ga4_coverage_pct ASC
LIMIT 5;
"""
print(con.execute(q_limit_clients).df())

print("\n==================================================")
print(" 3. TRAFFIC ATTRIBUTION DISCREPANCY (GSC Clicks vs GA4 Organic)")
print("==================================================")
q_limit_discrepancy = f"""
SELECT 
    SUM(gsc_clicks) AS total_gsc_clicks,
    SUM(sessions_organic) AS total_ga4_organic_sessions,
    SUM(gsc_clicks) - SUM(sessions_organic) AS click_session_delta
FROM '{local_file}'
WHERE ga4_data_available IS TRUE;
"""
print(con.execute(q_limit_discrepancy).df())

 1. GSC-ONLY vs. DUAL-INTEGRATION GAP
   gsc_only_rows  dual_integration_rows  gsc_only_pct
0      1718348.0               364347.0         17.46

 2. CLIENT-LEVEL UNBALANCED COVERAGE
            client_hash_id  total_records  ga4_records  ga4_coverage_pct
0  client_f6f0cdf26d03d7bd            520          0.0               0.0
1  client_2c32078d69f2cbad            341          0.0               0.0
2  client_0797ff3a1fc9a6a5           8060          0.0               0.0
3  client_400c21c81c8b46ef         106808          0.0               0.0
4  client_625b6439094e23e4         988497         37.0               0.0

 3. TRAFFIC ATTRIBUTION DISCREPANCY (GSC Clicks vs GA4 Organic)
   total_gsc_clicks  total_ga4_organic_sessions  click_session_delta
0          395204.0                    585408.0            -190204.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.